# Meeting Notes Agent

A LangGraph agent that takes a raw meeting transcript and turns it into structured notes:
action items (with owners), decisions made, and open questions.

Pipeline: `ingest -> extract_action_items -> extract_decisions -> extract_open_questions -> synthesize`

Each extraction node asks the model to reason through the transcript before committing to
an answer, and the reasoning is logged and printed alongside the extracted output.

In [29]:
from dotenv import load_dotenv

_ = load_dotenv()


In [30]:
from typing import TypedDict, List

from langchain_core.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver


## Model

Every node below calls this one `model` variable. Swapping providers only requires
changing this single line.

In [31]:
# Single point of model configuration - every node below uses this variable.
# Swap providers by changing only this line:
# model = ChatAnthropic(model="claude-opus-4-8")
model = ChatOpenAI(model="gpt-4o")


## State schema

In [32]:
class MeetingState(TypedDict):
    transcript: str
    action_items: str
    decisions: str
    open_questions: str
    reasoning_log: List[str]


## Sample meeting transcript

A fake weekly product sync with clear action items (some with owners), one explicit
decision, and a few questions the group left unresolved.

In [33]:
FAKE_TRANSCRIPT = """
Weekly Product Sync -- March 12, 2026

Attendees: Priya (Product Lead), Marcus (Eng Lead), Dana (Design), Tomas (QA)

Priya: Let's start with the onboarding flow redesign. Marcus, where are we on the backend work?

Marcus: The new signup API is done and deployed to staging. I still need to add rate limiting
before it goes to prod -- I'll get that done by Friday.

Priya: Great. Dana, how's the new onboarding screens looking?

Dana: Mostly done. I need one more round of feedback on the empty state illustrations. Priya,
can you review those by Wednesday?

Priya: Sure, I'll take a look Wednesday morning.

Marcus: One thing we need to decide -- are we launching the redesign to all users at once, or
doing a staged rollout?

Priya: Let's do a staged rollout. 10% of users in week one, then ramp up if there are no major
issues. Everyone good with that?

Dana: Agreed.

Tomas: Works for me, gives us time to catch bugs.

Priya: Okay, decided -- staged rollout starting at 10%.

Tomas: On the QA side, I ran the full regression suite against staging. Found two bugs -- one
where the "skip" button doesn't save state, and one where the illustration doesn't load on
slow connections. I'll file both today.

Marcus: Thanks, I'll pick up the skip button bug -- sounds like it's on the API side. Dana, the
image loading one is probably yours?

Dana: Yeah, I'll fix the lazy-loading issue by end of week.

Priya: One open question -- do we have a decision yet on whether the staged rollout applies to
enterprise customers too, or do we exclude them for now?

Marcus: I don't think we've discussed that. We should loop in the enterprise team before deciding.

Priya: Good call, let's flag that as open and follow up with them this week.

Dana: Also -- are we tracking a specific metric to decide whether to move from 10% to the next
stage? Or is it just "no major bugs"?

Priya: Good question, I don't think we've nailed that down. Let's figure out the success metric
before Friday.

Marcus: I can also ask -- do we need legal sign-off on the new data collection in the onboarding
flow? I honestly don't know the answer.

Priya: Also unresolved, let's add it to the list.

Priya: Alright, let's wrap up. Marcus -- rate limiting by Friday and the skip button bug. Dana --
illustration feedback loop and the lazy-load fix by end of week. Tomas -- file both bugs today.
I'll review the illustrations Wednesday and chase down the enterprise rollout question. Thanks
everyone.
"""


## Helper: parsing the reasoning / answer format

Every extraction node asks the model to respond with a `REASONING:` section followed by an
`ANSWER:` section. This helper splits the two apart.

In [34]:
def parse_response(content) -> tuple[str, str]:
    """Split a REASONING/ANSWER formatted model response into its two parts."""
    text = content if isinstance(content, str) else str(content)
    if "ANSWER:" in text:
        reasoning_part, answer_part = text.split("ANSWER:", 1)
        reasoning = reasoning_part.replace("REASONING:", "", 1).strip()
        answer = answer_part.strip()
    else:
        # Model didn't follow the format -- treat the whole thing as the answer.
        reasoning = "(no reasoning section returned)"
        answer = text.strip()
    return reasoning, answer


## Graph nodes

In [35]:
def ingest(state: MeetingState) -> MeetingState:
    """Normalize the raw transcript and reset the reasoning log."""
    cleaned = state["transcript"].strip()
    print("--- INGEST ---")
    print(f"Transcript received ({len(cleaned.split())} words).\n")
    return {"transcript": cleaned, "reasoning_log": []}


In [36]:
ACTION_ITEMS_PROMPT = """You are analyzing a meeting transcript to extract action items.

Think step by step through the transcript and identify every commitment, task, or follow-up
that someone agreed to do. Note who owns each item if a name is attached. Show your reasoning
first, then give the final answer.

Respond in exactly this format:
REASONING:
<your step-by-step reasoning>

ANSWER:
<bulleted list, one per line, in the form "- <task> (Owner: <name or "unspecified">)">

Transcript:
{transcript}
"""


def extract_action_items(state: MeetingState) -> MeetingState:
    prompt = ACTION_ITEMS_PROMPT.format(transcript=state["transcript"])
    response = model.invoke([HumanMessage(content=prompt)])
    reasoning, answer = parse_response(response.content)

    print("--- EXTRACT ACTION ITEMS ---")
    print("Reasoning:")
    print(reasoning)
    print("\nAction items:")
    print(answer, "\n")

    return {
        "action_items": answer,
        "reasoning_log": state["reasoning_log"] + [f"[action_items] {reasoning}"],
    }


In [37]:
DECISIONS_PROMPT = """You are analyzing a meeting transcript to extract decisions that were made.

Think step by step through the transcript and identify every explicit decision or agreement the
group reached -- something that was proposed and confirmed, not just discussed. Show your
reasoning first, then give the final answer.

Respond in exactly this format:
REASONING:
<your step-by-step reasoning>

ANSWER:
<bulleted list of decisions actually made, one per line>

Transcript:
{transcript}
"""


def extract_decisions(state: MeetingState) -> MeetingState:
    prompt = DECISIONS_PROMPT.format(transcript=state["transcript"])
    response = model.invoke([HumanMessage(content=prompt)])
    reasoning, answer = parse_response(response.content)

    print("--- EXTRACT DECISIONS ---")
    print("Reasoning:")
    print(reasoning)
    print("\nDecisions made:")
    print(answer, "\n")

    return {
        "decisions": answer,
        "reasoning_log": state["reasoning_log"] + [f"[decisions] {reasoning}"],
    }


In [38]:
OPEN_QUESTIONS_PROMPT = """You are analyzing a meeting transcript to extract open questions.

Think step by step through the transcript and identify every question, unresolved issue, or
item the group explicitly deferred or flagged for follow-up -- things that were raised but not
answered or decided. Show your reasoning first, then give the final answer.

Respond in exactly this format:
REASONING:
<your step-by-step reasoning>

ANSWER:
<bulleted list of open questions, one per line>

Transcript:
{transcript}
"""


def extract_open_questions(state: MeetingState) -> MeetingState:
    prompt = OPEN_QUESTIONS_PROMPT.format(transcript=state["transcript"])
    response = model.invoke([HumanMessage(content=prompt)])
    reasoning, answer = parse_response(response.content)

    print("--- EXTRACT OPEN QUESTIONS ---")
    print("Reasoning:")
    print(reasoning)
    print("\nOpen questions:")
    print(answer, "\n")

    return {
        "open_questions": answer,
        "reasoning_log": state["reasoning_log"] + [f"[open_questions] {reasoning}"],
    }


In [39]:
def synthesize(state: MeetingState) -> MeetingState:
    """Combine the three extracted sections into one structured report."""
    report = f"""MEETING SUMMARY
================

ACTION ITEMS
------------
{state['action_items']}

DECISIONS MADE
--------------
{state['decisions']}

OPEN QUESTIONS
--------------
{state['open_questions']}
"""
    print("--- SYNTHESIZE ---")
    print(report)

    return {
        "reasoning_log": state["reasoning_log"] + ["[synthesize] Combined all sections into the final report."]
    }


## Build the graph

Five nodes, wired sequentially.

In [40]:
builder = StateGraph(MeetingState)

builder.add_node("ingest", ingest)
builder.add_node("extract_action_items", extract_action_items)
builder.add_node("extract_decisions", extract_decisions)
builder.add_node("extract_open_questions", extract_open_questions)
builder.add_node("synthesize", synthesize)

builder.add_edge(START, "ingest")
builder.add_edge("ingest", "extract_action_items")
builder.add_edge("extract_action_items", "extract_decisions")
builder.add_edge("extract_decisions", "extract_open_questions")
builder.add_edge("extract_open_questions", "synthesize")
builder.add_edge("synthesize", END)


## Run it

Checkpointing with `SqliteSaver` is carried over from the persistence lesson earlier in this
course -- using an in-memory database here since this is just a demo run, not something that
needs to survive across processes.

In [41]:
with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    app = builder.compile(checkpointer=checkpointer)

    config = {"configurable": {"thread_id": "meeting-demo-1"}}
    initial_state: MeetingState = {
        "transcript": FAKE_TRANSCRIPT,
        "action_items": "",
        "decisions": "",
        "open_questions": "",
        "reasoning_log": [],
    }

    result = app.invoke(initial_state, config=config)


--- INGEST ---
Transcript received (435 words).

--- EXTRACT ACTION ITEMS ---
Reasoning:
1. Marcus mentioned he needs to add rate limiting to the signup API before it goes to production, which is a clear task, and he committed to doing it by Friday.
2. Priya agreed to review the empty state illustrations by Wednesday morning, which is a task she committed to.
3. Priya decided on a staged rollout for the redesign and confirmed it as a team action item.
4. Tomas agreed to file the QA bugs he found today. It's a task that he takes ownership of.
5. Marcus agreed to pick up the bug related to the "skip" button not saving state, which Tomas identified in the regression.
6. Dana agreed to fix the lazy-loading issue with illustrations and committed to completing it by the end of the week.
7. Priya mentioned needing to follow up with the enterprise team regarding whether the staged rollout applies to enterprise customers.
8. Priya assigned the task of figuring out the success metric for moving 

## Final structured output

In [42]:
print("ACTION ITEMS")
print(result["action_items"])
print()
print("DECISIONS MADE")
print(result["decisions"])
print()
print("OPEN QUESTIONS")
print(result["open_questions"])


ACTION ITEMS
- Add rate limiting to the signup API by Friday (Owner: Marcus)
- Review empty state illustrations by Wednesday morning (Owner: Priya)
- Conduct a staged rollout starting at 10% (Owner: Unspecified)
- File two bugs found during regression today (Owner: Tomas)
- Fix "skip" button bug (Owner: Marcus)
- Fix lazy-loading issue with illustrations by the end of the week (Owner: Dana)
- Follow up with the enterprise team regarding staged rollout for enterprise customers (Owner: Priya)
- Define success metric for moving from 10% rollout to next stage by Friday (Owner: Unspecified)
- Determine if legal sign-off is needed for new data collection in onboarding flow (Owner: Unspecified)

DECISIONS MADE
- Staged rollout for the onboarding flow redesign starting at 10% of users in week one.

OPEN QUESTIONS
- Do we have a decision on whether the staged rollout applies to enterprise customers, or do we exclude them for now?
- Are we tracking a specific metric to decide whether to move fro